# **Laboratorio 1**

Laura Sanchez Bernal - 202411353

Baruc ....

## **Actividades a realizar**

2. Justificando las decisiones tomadas con base en los resultados obtenidos en el paso anterior y de acuerdo con el modelo que van a construir.

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

from sklearn.impute import SimpleImputer

datos = pd.read_csv('./data/Datos lab 1.csv', )
datosPrueba = pd.read_csv('./data/Datos Test Lab 1.csv', )

data = datos.copy()
dataPrueba = datosPrueba.copy()

In [16]:
from importlib.metadata import version
print(f"Versión de Pandas: {version('pandas')}")
print(f"Versión de Matplotlib: {version('matplotlib')}")
print(f"Versión de Seaborn: {version('seaborn')}")
print(f"Versión de Scikit learn: {version('scikit-learn')}")

Versión de Pandas: 3.0.3
Versión de Matplotlib: 3.11.0
Versión de Seaborn: 0.13.2
Versión de Scikit learn: 1.9.0


# **Unicidad**

1. **Filas duplicadas:** 4 filas idénticas en las 27 columnas, es un caso sin ambigüedad. Se conserva solo la primera con drop_duplicates()


In [17]:
n_duplicados = int(data.duplicated(keep=False).sum())
print(f"Número de registros duplicados: {n_duplicados}")
data = data.drop_duplicates(keep='first')


Número de registros duplicados: 8


# **Consistencia**

1. **Sector_viento y mes:** Ambas columnas presentaban muchas variantes distintas para las mismas categorías como mezcla de mayúsculas/minúsculas, español/inglés, abreviado/completo. Por ejemplo, 27 categorías para sector_viento cuando solo deberían existir 8. Aquí el problema es complejo puesto que hay traducción de idioma y formato abreviado, así que se hizo un diccionario de mapeo ya que es la estrategia que el notebook recomienda para "múltiples variantes a unificar". Se usó .replace() en lugar de .map() porque .replace() conserva sin cambios cualquier valor que no esté explícitamente en el diccionario, mientras que .map() lo convertiría en NaN. Esto es más seguro si existiera alguna variante que no detectamos en la exploración inicial, evitando pérdida silenciosa de información.

In [18]:
data['sector_viento'].value_counts()

mapeo_sector_viento = {
    'so': 'SO', 'Suroeste': 'SO', 'SOUTHWEST': 'SO',
    's': 'S', 'South': 'S', 'SOUTH': 'S', 'Sur': 'S',
    'ne': 'NE', 'NORTHEAST': 'NE', 'Noreste': 'NE',
    'o': 'O', 'West': 'O', 'WEST': 'O',
    'North': 'N', 'no': 'N', 'Norte': 'N',
    'Este': 'E',
    'SOUTHEAST': 'SE',
    'NORTHWEST': 'NO',
}
data['sector_viento'] = data['sector_viento'].replace(mapeo_sector_viento)
data['sector_viento'].value_counts()

sector_viento
SO          774
S           580
NE          471
O           325
N           137
E            79
SE           69
NO           43
NORTH         6
East          4
EAST          4
Sureste       3
Oeste         3
Noroeste      3
Name: count, dtype: int64

In [19]:
data['mes'].value_counts()

mapeo_mes = {
    'Enero': 'enero', 'january': 'enero', 'January': 'enero',
    'Febrero': 'febrero', 'february': 'febrero', 'February': 'febrero',
    'Marzo': 'marzo', 'march': 'marzo', 'March': 'marzo',
    'Abril': 'abril', 'april': 'abril', 'April': 'abril',
    'Mayo': 'mayo', 'may': 'mayo', 'May': 'mayo',
    'Junio': 'junio', 'june': 'junio', 'June': 'junio',
    'Julio': 'julio', 'july': 'julio', 'July': 'julio', 'JULY': 'julio',
    'Agosto': 'agosto', 'august': 'agosto', 'August': 'agosto',
    'Septiembre': 'septiembre', 'september': 'septiembre', 'September': 'septiembre',
    'Octubre': 'octubre', 'october': 'octubre', 'October': 'octubre',
    'Noviembre': 'noviembre', 'november': 'noviembre', 'November': 'noviembre',
    'Diciembre': 'diciembre', 'december': 'diciembre', 'December': 'diciembre',
}
data['mes'] = data['mes'].replace(mapeo_mes)
data['mes'].value_counts()

mes
enero         201
julio         189
mayo          177
diciembre     172
octubre       165
agosto        165
marzo         165
septiembre    156
noviembre     151
febrero       149
junio         146
abril         126
NOVEMBER       32
APRIL          29
Feb            29
SEPTEMBER      28
Nov            28
MARCH          28
Jun            28
Sep            25
Sept           25
Aug            25
Apr            25
Dec            22
MAY            21
Mar            20
DECEMBER       20
Jul            20
Oct            19
AUGUST         19
FEBRUARY       19
JUNE           18
JANUARY        16
Jan            13
OCTOBER        13
Name: count, dtype: int64

# **Validez**

1. **Fila corrupta 2015-07-13**: Afecta varias columnas a la vez con valores imposibles como (-9999, -1250, NaN), y no contamos con una regla que permita reconstruirla. Al no poder corregirla, se elimina para no inventar datos.

In [20]:
fila_corrupta = data[data['rafaga_min'] == -9999].copy()
print(f"Filas corruptas a eliminar: {fila_corrupta.shape[0]}")
data = data[data['rafaga_min'] != -9999]

Filas corruptas a eliminar: 1


2. **Presion_media:** Los valores >1050 son físicamente imposibles, pero al dividirlos entre 10 caen dentro del rango normal, puede ser un error de escritura y no creemos que sea un dato perdido. Usamoa .loc porque es una regla de negocio conocida, aplicable solo a los registros afectados.


In [21]:
mask_presion = (data['presion_media'] > 1050)
print(f"Registros corregidos en presion_media: {mask_presion.sum()}")
data.loc[mask_presion, 'presion_media'] = data.loc[mask_presion, 'presion_media'] / 10


Registros corregidos en presion_media: 5


3. **Humedad_media** Los valores <1 están en escala de proporción (0-1) en vez de porcentaje (0-100). Desidimos multiplicar por 100, pues recupera el dato real. Igual que con presión, es una corrección puntual con regla conocida, no una imputación.

In [22]:
mask_humedad = (data['humedad_media'] < 1)
print(f"Registros corregidos en humedad_media: {mask_humedad.sum()}")
data.loc[mask_humedad, 'humedad_media'] = data.loc[mask_humedad, 'humedad_media'] * 100

Registros corregidos en humedad_media: 775


4. **Registros_del_dia > 144:** Se encontraron 4 registros que superan el máximo lógico de 144 mediciones diarias. Al no poder determinarse la causa exacta del exceso de mediciones y no tratarse de columnas correlacionadas que permitan inferir el valor correcto, como sí pasaba con presión y humedad, se optó por eliminarlos en vez de intentar adivinar cuál sería el conteo correcto.

In [23]:
fuera_rango_registros = data[data['registros_del_dia'] > 144].copy()
print(f"Registros fuera de rango en registros_del_dia: {fuera_rango_registros.shape[0]}")
data = data[data['registros_del_dia'] <= 144]

Registros fuera de rango en registros_del_dia: 4


# **Completitud**

Todas las columnas tienen menos del 5% de nulos, dentro del rango donde el notebook recomienda imputación simple en lugar de eliminación masiva. Se usa mediana para las variables cuantitativas (ya sabemos que el dataset tuvo outliers, así que la mediana es más robusta que la media) y moda para las nominales, siguiendo la tabla de estrategias por tipo de variable. La columna fecha es la única excepción: al ser un identificador temporal sin valor numérico ni categórico que imputar, se eliminan sus filas nulas en vez de inventar una fecha.

In [24]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)

temp_max_manana      0.036948
viento_min           0.034940
mes                  0.034538
estacion_anio        0.034538
anio                 0.034137
humedad_max          0.033333
viento_media         0.032530
presion_desv         0.031325
direccion_viento     0.031325
viento_norte         0.031325
rafaga_desv          0.030924
humedad_min          0.030924
viento_max           0.030924
rafaga_media         0.030522
humedad_media        0.029317
rafaga_min           0.028916
presion_media        0.028916
fecha                0.028916
rafaga_max           0.028514
presion_min          0.028112
sector_viento        0.028112
viento_desv          0.027309
dia_del_anio         0.026104
viento_este          0.025703
humedad_desv         0.024498
presion_max          0.024096
registros_del_dia    0.000000
dtype: float64

In [25]:
cuantitativas = [
    'presion_media', 'presion_min', 'presion_max', 'presion_desv',
    'humedad_media', 'humedad_min', 'humedad_max', 'humedad_desv',
    'viento_media', 'viento_min', 'viento_max', 'viento_desv',
    'rafaga_media', 'rafaga_min', 'rafaga_max', 'rafaga_desv',
    'viento_norte', 'viento_este', 'direccion_viento', 'temp_max_manana',
    'anio', 'dia_del_anio', 'registros_del_dia'
]

imputer_mediana = SimpleImputer(strategy='median')
data[cuantitativas] = imputer_mediana.fit_transform(data[cuantitativas])


In [26]:
cualitativas = ['estacion_anio', 'mes', 'sector_viento']

imputer_moda = SimpleImputer(strategy='most_frequent')
data[cualitativas] = imputer_moda.fit_transform(data[cualitativas])

In [27]:
filas_antes = data.shape[0]
data = data.dropna(subset=['fecha'])
print(f"Filas eliminadas por fecha nula: {filas_antes - data.shape[0]}")

Filas eliminadas por fecha nula: 72


In [28]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)

fecha                0.0
rafaga_min           0.0
sector_viento        0.0
mes                  0.0
estacion_anio        0.0
dia_del_anio         0.0
anio                 0.0
registros_del_dia    0.0
direccion_viento     0.0
viento_este          0.0
viento_norte         0.0
rafaga_desv          0.0
rafaga_max           0.0
rafaga_media         0.0
presion_media        0.0
viento_desv          0.0
viento_max           0.0
viento_min           0.0
viento_media         0.0
humedad_desv         0.0
humedad_max          0.0
humedad_min          0.0
humedad_media        0.0
presion_desv         0.0
presion_max          0.0
presion_min          0.0
temp_max_manana      0.0
dtype: float64

# **Resumen de Limpieza:**


In [ ]:
print("=== Dimensiones del dataset ===")
print(f"Filas: {data.shape[0]}, Columnas: {data.shape[1]}")

print("\n=== Valores nulos ===")
print(data.isnull().sum()[data.isnull().sum() > 0])
if data.isnull().sum().sum() == 0:
    print("Sin valores nulos")

print("\n=== Duplicados ===")
dup = data.duplicated().sum()
print(f"Filas duplicadas: {dup}")
if dup == 0:
    print("Sin duplicados")

print("\n=== Valores únicos en sector_viento ===")
print(data['sector_viento'].value_counts())

print("\n=== Valores únicos en mes ===")
print(data['mes'].value_counts())

print("\n=== Rango de presion_media ===")
print(f"Min: {data['presion_media'].min()}, Max: {data['presion_media'].max()}")
if data['presion_media'].min() >= 950 and data['presion_media'].max() <= 1050:
    print("presion_media dentro del rango esperado")

print("\n=== Rango de humedad_media ===")
print(f"Min: {data['humedad_media'].min()}, Max: {data['humedad_media'].max()}")
if data['humedad_media'].min() >= 0 and data['humedad_media'].max() <= 100:
    print("humedad_media dentro del rango [0, 100]")

print("\n=== Rango de registros_del_dia ===")
print(f"Min: {data['registros_del_dia'].min()}, Max: {data['registros_del_dia'].max()}")
if data['registros_del_dia'].max() <= 144:
    print("registros_del_dia dentro del máximo lógico 144")

=== Dimensiones del dataset ===
Filas: 2418, Columnas: 27

=== Valores nulos ===
Series([], dtype: int64)
Sin valores nulos

=== Duplicados ===
Filas duplicadas: 0
Sin duplicados

=== Valores únicos en sector_viento ===
sector_viento
SO          800
S           541
NE          447
O           304
N           126
E            71
SE           65
NO           42
NORTH         6
EAST          4
Sureste       3
Oeste         3
Noroeste      3
East          3
Name: count, dtype: int64

=== Valores únicos en mes ===
mes
enero         275
julio         180
mayo          168
diciembre     162
octubre       154
agosto        154
marzo         153
septiembre    147
noviembre     144
febrero       136
junio         134
abril         117
NOVEMBER       29
APRIL          27
Nov            27
MARCH          27
Jun            27
Feb            27
SEPTEMBER      26
Aug            25
Apr            25
Sep            24
Sept           24
Dec            21
Mar            20
DECEMBER       19
Jul          